In [13]:
import pandas as pd, numpy as np
from scipy.stats import norm

sales = pd.read_csv("../data/processed/fact_sales_clean.csv", parse_dates=["date"])
product = pd.read_csv("../data/raw/dim_product.csv")
supplier = pd.read_csv("../data/raw/dim_supplier.csv")

# Per store-product demand stats
demand_stats = sales.groupby(["store_id","product_id"])["quantity"].agg(["mean","std"]).reset_index()
demand_stats.columns = ["store_id","product_id","D_avg","sigma_D"]

merged = demand_stats.merge(product[["product_id","primary_supplier_id","cost_price"]], on="product_id")
merged = merged.merge(supplier[["supplier_id","base_lead_time_days"]],
                       left_on="primary_supplier_id", right_on="supplier_id")

Z = norm.ppf(0.95)  # 1.645 for 95% service level
merged["safety_stock"] = Z * merged["sigma_D"] * np.sqrt(merged["base_lead_time_days"])
merged["reorder_point"] = merged["D_avg"] * merged["base_lead_time_days"] + merged["safety_stock"]

# EOQ
ORDERING_COST = 500   # assumption — document this
HOLDING_COST_RATE = 0.15  # 15% of unit cost per year, common assumption
merged["annual_demand"] = merged["D_avg"] * 365
merged["holding_cost"] = merged["cost_price"] * HOLDING_COST_RATE
merged["EOQ"] = np.sqrt((2 * merged["annual_demand"] * ORDERING_COST) / merged["holding_cost"])

merged.to_csv("../data/processed/inventory_recommendations.csv", index=False)
print(merged[["store_id","product_id","safety_stock","reorder_point","EOQ"]].head(10))

  store_id product_id  safety_stock  reorder_point         EOQ
0     S001       P001     23.657754     114.860189  229.864623
1     S001       P002      8.167430      35.034097   25.078257
2     S001       P003     14.224715      63.389080  173.633321
3     S001       P004      3.308214       9.557349   49.678311
4     S001       P005      9.445712      41.581535   17.688629
5     S001       P006     11.956380      57.082753  126.911291
6     S001       P007      7.169661      24.234222  144.875017
7     S001       P008     11.401690      53.901690  101.884944
8     S001       P009     10.101338      45.621338  193.570798
9     S001       P010      8.401918      32.900545  121.343510


In [14]:
import pulp

# Simplify: pick top N SKU-store combos below reorder point, decide order qty per combo
candidates = merged.copy()
candidates = candidates.merge(product[["product_id","selling_price"]], on="product_id")
candidates["unit_margin"] = candidates["selling_price"] - candidates["cost_price"]

BUDGET = 2_000_000       # ₹20L
WAREHOUSE_CAPACITY = 100_000  # units

prob = pulp.LpProblem("OptiChain_Order_Optimization", pulp.LpMaximize)

# Decision variable: order quantity for each candidate, bounded by EOQ as a practical cap
order_vars = {
    i: pulp.LpVariable(f"order_{i}", lowBound=0, upBound=row.EOQ * 2)
    for i, row in candidates.iterrows()
}

# Objective: maximize expected profit from ordered units
prob += pulp.lpSum(order_vars[i] * candidates.loc[i, "unit_margin"] for i in candidates.index)

# Constraint: total cost within budget
prob += pulp.lpSum(order_vars[i] * candidates.loc[i, "cost_price"] for i in candidates.index) <= BUDGET

# Constraint: total units within warehouse capacity
prob += pulp.lpSum(order_vars[i] for i in candidates.index) <= WAREHOUSE_CAPACITY

prob.solve()

candidates["recommended_order_qty"] = [order_vars[i].value() for i in candidates.index]
candidates[["store_id","product_id","recommended_order_qty"]].to_csv("../data/processed/order_recommendations.csv", index=False)
print("Status:", pulp.LpStatus[prob.status])
print(candidates[candidates.recommended_order_qty > 0][["store_id","product_id","recommended_order_qty","unit_margin"]].head(15))

Status: Optimal
    store_id product_id  recommended_order_qty  unit_margin
108     S002       P049              458.69901       509.12
348     S006       P049              465.02915       509.12
468     S008       P049              386.28397       509.12
648     S011       P049              441.99172       509.12


In [15]:
total_cost = (
    candidates["recommended_order_qty"] * candidates["cost_price"]
).sum()

total_units = candidates["recommended_order_qty"].sum()

print("Total Order Cost: ₹", round(total_cost, 2))
print("Budget Limit: ₹", BUDGET)

print("\nTotal Units Ordered:", round(total_units, 2))
print("Warehouse Capacity:", WAREHOUSE_CAPACITY)

print("\nBudget Used:", round(total_cost / BUDGET * 100, 2), "%")
print("Warehouse Capacity Used:", round(total_units / WAREHOUSE_CAPACITY * 100, 2), "%")

Total Order Cost: ₹ 1999999.99
Budget Limit: ₹ 2000000

Total Units Ordered: 1752.0
Warehouse Capacity: 100000

Budget Used: 100.0 %
Warehouse Capacity Used: 1.75 %
